# Notebook 29 — Invariant causal prediction for pathway relationships

`InvariantPathwayPredictor` searches for the subset of candidate
parents whose regression residuals are *invariant* across
environments. Under ICP assumptions the identifiable causal parents
are the intersection of all invariant subsets.

Research use only. Not for clinical decision-making.

In [ ]:
import numpy as np
import pandas as pd

from pathway_subtyping.causal import InvariantPathwayPredictor

rng = np.random.default_rng(0)
n_per_env = 300

def gen(mu_x1, mu_x2, sig_x1, sig_x2, mu_x4):
    X1 = rng.standard_normal(n_per_env) * sig_x1 + mu_x1
    X2 = rng.standard_normal(n_per_env) * sig_x2 + mu_x2
    Y  = 0.8 * X1 + 0.6 * X2 + 0.2 * rng.standard_normal(n_per_env)
    X3 = 0.7 * Y + rng.standard_normal(n_per_env) * 0.4
    X4 = rng.standard_normal(n_per_env) + mu_x4
    return pd.DataFrame({'X1': X1, 'X2': X2, 'X3': X3, 'X4': X4, 'Y': Y})

a = gen(0.0, 0.0, 1.0, 1.0, 0.0); a['env'] = 'A'
b = gen(2.0, -1.5, 2.0, 0.4, -2.0); b['env'] = 'B'
df = pd.concat([a, b], ignore_index=True)
df.head()

## 1. Identify the causal parents of Y

In [ ]:
predictor = InvariantPathwayPredictor(alpha=0.05, max_subset_size=3)
report = predictor.fit(
    X=df[['X1', 'X2', 'X3', 'X4']],
    y=df['Y'],
    target_name='Y',
    environments=df['env'].to_numpy(),
)
print(report.summary())

## 2. Precision / recall vs ground truth

Ground-truth parents (by construction): X1 and X2. X3 is a child of
Y; X4 is an environment-shifted noise variable.

In [ ]:
print(f'recall against ground truth: {report.recall_against(["X1", "X2"]):.2f}')
print(f'precision against ground truth: {report.precision_against(["X1", "X2"]):.2f}')
print(f'identifiable: {sorted(report.identifiable_parents)}')

## See also

- Peters, Buhlmann, Meinshausen (2016). *Causal inference using
  invariant prediction.* JRSS-B.
- PSF v0.6 roadmap — Phase 3 F11: [docs/roadmap-v06-codeberg.md](../../docs/roadmap-v06-codeberg.md)